# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, find_all_hrs_ft_files, detect_app_version,
    # File writer
    write_hrs2,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    make_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
    FT_SNAP_HZ, ft_snap_hz, ft_stim_adc_hz, compute_ft_trial_hz,
    load_all_recordings, make_ft_viewer,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [2]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [

    ("HRPILOT-17 ControlM5",  "HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26",  10000.0),

    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

1 recording(s) configured.
  [0] 'HRPILOT-17 ControlM5'  →  HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26  (sample_rate=10000.0 Hz)


# Section 7: Trial Cleaning — Exclude Bad Trials & Export

Filter out unwanted trials using one or more of the methods below, preview the result, then write the cleaned data to a new binary file in the same format as the original.

**Available filter methods (combine freely — a trial is excluded if ANY rule rejects it):**
- **Method 1 — Trial number**: exclude specific trial numbers (1-based) or ranges
- **Method 2 — M-wave MRA**: keep only trials where the stored M-wave MRA falls within [M_MRA_MIN, M_MRA_MAX]
- **Method 3 — H-wave MRA**: keep only trials where the stored H-wave MRA falls within [H_MRA_MIN, H_MRA_MAX]
- **Method 4 — Stimulation amplitude**: keep only trials within [STIM_MIN_MA, STIM_MAX_MA]

In [ ]:
# ── Trial Cleaning Configuration ────────────────────────────────────────────────
# Select which recording and stage to clean.
RECORDING_TO_CLEAN = _active_rec_label   # or a specific label from RECORDING_DIRS
STAGE_TO_CLEAN     = list(_all_recordings[_active_rec_label]['stage_map'].keys())[0]
                    # e.g. 'control_mode', 'mh_recruitment', 'dcp' — see stage summary above

# ── Method 1: Exclude by trial number (1-based) ─────────────────────────────────
EXCLUDE_TRIAL_NUMS   = []           # e.g. [3, 7, 12]       — individual trial numbers
EXCLUDE_TRIAL_RANGES = [(643,777)]           # e.g. [(1, 5), (20, 30)] — inclusive ranges

# ── Method 2: M-wave MRA window (µV) — None = no limit ─────────────────────────
M_MRA_MIN = None     # keep trials with M-wave MRA >= this
M_MRA_MAX = None     # keep trials with M-wave MRA <= this

# ── Method 3: H-wave MRA window (µV) — None = no limit ─────────────────────────
H_MRA_MIN = None
H_MRA_MAX = None

# ── Method 4: Stimulation amplitude window (mA) — None = no limit ──────────────
STIM_MIN_MA = None
STIM_MAX_MA = None

print(f"Will clean: {RECORDING_TO_CLEAN!r} → stage {STAGE_TO_CLEAN!r}")
_rec_clean = _all_recordings[RECORDING_TO_CLEAN]
_st_clean, _sh_clean, _se_clean, _slbl_clean = _rec_clean['stage_map'][STAGE_TO_CLEAN]
print(f"Total trials in stage: {len(_st_clean)}")

Will clean: 'HRPILOT-17 ControlM5' → stage 'control_mode'
Total trials in stage: 1026


In [ ]:
# ── Apply Filters — Preview ─────────────────────────────────────────────────────
# Run this cell to see exactly which trials will be kept and excluded.
# Nothing is written yet.

_excl_num_set = set(EXCLUDE_TRIAL_NUMS)

def _trial_exclusion_reason(trial, trial_num_1based):
    """Return a reason string if the trial should be excluded, else ''."""
    reasons = []

    # Method 1: trial number
    if trial_num_1based in _excl_num_set:
        reasons.append(f'number {trial_num_1based} in EXCLUDE_TRIAL_NUMS')
    for lo, hi in EXCLUDE_TRIAL_RANGES:
        if lo <= trial_num_1based <= hi:
            reasons.append(f'number {trial_num_1based} in range ({lo}–{hi})')

    # Method 2: M-wave MRA
    m_mra = getattr(trial, 'm_wave_mra', None)
    if m_mra is not None:
        if M_MRA_MIN is not None and m_mra < M_MRA_MIN:
            reasons.append(f'M-MRA {m_mra:.2f} < {M_MRA_MIN}')
        if M_MRA_MAX is not None and m_mra > M_MRA_MAX:
            reasons.append(f'M-MRA {m_mra:.2f} > {M_MRA_MAX}')

    # Method 3: H-wave MRA
    h_mra = getattr(trial, 'h_wave_mra', None)
    if h_mra is not None:
        if H_MRA_MIN is not None and h_mra < H_MRA_MIN:
            reasons.append(f'H-MRA {h_mra:.2f} < {H_MRA_MIN}')
        if H_MRA_MAX is not None and h_mra > H_MRA_MAX:
            reasons.append(f'H-MRA {h_mra:.2f} > {H_MRA_MAX}')

    # Method 4: stim amplitude
    amp = getattr(trial, 'stimulation_amplitude_ma', None)
    if amp is not None:
        if STIM_MIN_MA is not None and amp < STIM_MIN_MA:
            reasons.append(f'stim {amp:.3f} mA < {STIM_MIN_MA}')
        if STIM_MAX_MA is not None and amp > STIM_MAX_MA:
            reasons.append(f'stim {amp:.3f} mA > {STIM_MAX_MA}')

    return '; '.join(reasons)


def apply_trial_filters(trials, emg_blocks):
    """Return (kept_trials, kept_emg, excluded_info).
    excluded_info: list of (trial_num_1based, trial, reason_str)
    """
    kept_t, kept_e, excl = [], [], []
    for i, t in enumerate(trials):
        num = i + 1
        reason = _trial_exclusion_reason(t, num)
        if reason:
            excl.append((num, t, reason))
        else:
            kept_t.append(t)
            if emg_blocks and i < len(emg_blocks):
                kept_e.append(emg_blocks[i])
    return kept_t, kept_e, excl


_kept_trials, _kept_emg, _excluded_info = apply_trial_filters(_st_clean, _se_clean)

print(f"Recording : {RECORDING_TO_CLEAN!r}")
print(f"Stage     : {_slbl_clean}")
print(f"Total     : {len(_st_clean)} trials")
print(f"Keep      : {len(_kept_trials)} trials")
print(f"Exclude   : {len(_excluded_info)} trials")

if _excluded_info:
    print(f"\n{'Trial':>6}  {'Stim (mA)':>10}  {'M-MRA (µV)':>12}  {'H-MRA (µV)':>12}  Reason")
    print("─" * 80)
    for num, t, reason in _excluded_info:
        amp   = getattr(t, 'stimulation_amplitude_ma', float('nan'))
        m_mra = getattr(t, 'm_wave_mra',               float('nan'))
        h_mra = getattr(t, 'h_wave_mra',               float('nan'))
        print(f"{num:>6}  {amp:>10.3f}  {m_mra:>12.2f}  {h_mra:>12.2f}  {reason}")
else:
    print("\nNo trials excluded — all pass the current filter criteria.")

Recording : 'HRPILOT-17 ControlM5'
Stage     : Control Mode (.hrs2)
Total     : 1026 trials
Keep      : 891 trials
Exclude   : 135 trials

 Trial   Stim (mA)    M-MRA (µV)    H-MRA (µV)  Reason
────────────────────────────────────────────────────────────────────────────────
   643       0.140           nan           nan  number 643 in range (643–777)
   644       0.140           nan           nan  number 644 in range (643–777)
   645       0.140           nan           nan  number 645 in range (643–777)
   646       0.140           nan           nan  number 646 in range (643–777)
   647       0.140           nan           nan  number 647 in range (643–777)
   648       0.140           nan           nan  number 648 in range (643–777)
   649       0.140           nan           nan  number 649 in range (643–777)
   650       0.140           nan           nan  number 650 in range (643–777)
   651       0.140           nan           nan  number 651 in range (643–777)
   652       0.140     

In [ ]:
# ── Write Cleaned File ─────────────────────────────────────────────────────────
# Run AFTER reviewing the preview above.
# The output file is in the exact same binary format as the input.
# Default: saves a new file in the same directory with '_cleaned' appended to the stem.

import os as _os

# Find the original file path for the selected stage
_rdir_clean = next(d for lbl, d, _ in RECORDING_DIRS if lbl == RECORDING_TO_CLEAN)
_rp1c, _rp2c, _rp3c, _rp4c, _rp5c, _rp6c, _rpftc = find_hrs_files(_rdir_clean)
_stage_src_path = {
    'mh_recruitment': _rp1c,
    'control_mode':   _rp2c,
    'dcp':            _rp3c,
    'up_cond_pellet': _rp4c,
    'down_cond_vns':  _rp5c,
    'up_cond_vns':    _rp6c,
}[STAGE_TO_CLEAN]

if _stage_src_path is None:
    print(f"ERROR: source file for stage {STAGE_TO_CLEAN!r} not found in {_rdir_clean!r}")
else:
    _stem, _ext = _os.path.splitext(_stage_src_path)
    OUTPUT_PATH = _stem + '_cleaned' + _ext   # ← change this to override the output path

    if len(_kept_trials) == 0:
        print("WARNING: 0 trials kept — nothing to write. Adjust your filters.")
    elif _os.path.abspath(OUTPUT_PATH) == _os.path.abspath(_stage_src_path):
        print("ERROR: OUTPUT_PATH is the same as the source file. Change OUTPUT_PATH to avoid overwriting.")
    else:
        write_hrs2(OUTPUT_PATH, _sh_clean, _kept_trials, _kept_emg)
        print(f"Wrote {len(_kept_trials)} / {len(_st_clean)} trials to:")
        print(f"  {OUTPUT_PATH}")
        print(f"\nExcluded {len(_excluded_info)} trial(s). Original file is unchanged.")

Wrote 891 / 1026 trials to:
  HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26\BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26_20260826T101805_cleaned.hrs2

Excluded 135 trial(s). Original file is unchanged.


# Section 8: Convert Conditioning Stage File → .hrs2

Reads any conditioning stage file (`.hrs3`, `.hrs4`, `.hrs5`, `.hrs6`) and writes it as a `.hrs2` (Control Mode) binary file.

**Use case:** a session was recorded using a conditioning stage but the animal was performing control-mode-like behaviour, and you want the data analysed as plain peri-stimulus trials in `Read_H-Reflex_App.ipynb`.

**What is preserved:**
- All raw EMG trial data, sync data, timestamps, stim amplitude, onset indices, background EMG, polarity
- M-wave response, H-wave response, H:M ratio, and M-wave stabilisation fields (if the source file_version ≥ 2)

**What is discarded:**
- Conditioning-specific metadata only: `success_threshold`, `is_success`, `pellet_delivered` / `vns_delivered`, `aux_flag` — these have no equivalent in `.hrs2` and are not needed for EMG analysis.

After writing, add the output file's directory to `RECORDING_DIRS` in `Read_H-Reflex_App.ipynb` with extension `.hrs2`.

In [ ]:
# ── Convert Conditioning Stage File → .hrs2 ───────────────────────────────────
# Reads a .hrs3 / .hrs4 / .hrs5 / .hrs6 file and writes it as a .hrs2 binary.
# All EMG trial data is preserved.  Conditioning-only fields are silently ignored.
#
# ── Configuration ─────────────────────────────────────────────────────────────
import os as _os, copy as _copy, math as _math

SOURCE_PATH = (
    "HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26/"
    "BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26_20260828T102534.hrs4"
)

# Output path — defaults to same directory and stem, extension changed to .hrs2.
# Override this to write elsewhere, e.g.:
#   OUTPUT_PATH = "HRPilot-17_Control/BASELINE1_.../BASELINE1_cleaned.hrs2"
_src_stem, _src_ext = _os.path.splitext(SOURCE_PATH)
OUTPUT_PATH = _src_stem + '.hrs2'

# Stage label written into the output header (shown in notebooks / summaries).
# None = keep whatever is in the source file.
STAGE_NAME = "Control Mode"
STAGE_DESC = None   # None → keep source description unchanged

# ── Reader map — add new extensions here if needed ───────────────────────────
_EXT_TO_READER = {
    '.hrs3': read_hrs3,
    '.hrs4': read_hrs4,
    '.hrs5': read_hrs5,
    '.hrs6': read_hrs6,
}

# ── Validation ────────────────────────────────────────────────────────────────
_ext = _src_ext.lower()
_ok  = True

if _ext not in _EXT_TO_READER:
    print(f"ERROR: unsupported source extension {_src_ext!r}.")
    print(f"       Supported extensions: {list(_EXT_TO_READER.keys())}")
    _ok = False
elif not _os.path.isfile(SOURCE_PATH):
    print(f"ERROR: source file not found:\n  {SOURCE_PATH}")
    _ok = False
elif _os.path.abspath(OUTPUT_PATH) == _os.path.abspath(SOURCE_PATH):
    print("ERROR: OUTPUT_PATH is the same as SOURCE_PATH — choose a different output path.")
    _ok = False

if _ok:
    # ── Read ─────────────────────────────────────────────────────────────────
    print(f"Reading {_src_ext!r}: {SOURCE_PATH}")
    _src_hdr, _src_trials, _src_emg = _EXT_TO_READER[_ext](SOURCE_PATH)
    print(f"  {len(_src_trials)} trials  |  source file_version={_src_hdr.file_version}"
          f"  |  stage: {_src_hdr.stage_name!r}")
    print(f"  sample_rate={_src_hdr.sample_rate} Hz  |  subject: {_src_hdr.subject_id!r}")

    # ── Build output header ───────────────────────────────────────────────────
    _out_hdr = _copy.deepcopy(_src_hdr)
    if STAGE_NAME is not None:
        _out_hdr.stage_name = STAGE_NAME
    if STAGE_DESC is not None:
        _out_hdr.stage_description = STAGE_DESC

    # Choose output file_version:
    #   14 → Control Mode block (6) with M/H response + M-wave stabilizer fields, and the
    #        header's filtering_protocol / booth_snapshot / filter_config (fv9 dropped
    #        these, so OFFLINE files lost their OFFLINE rebuild when re-loaded).
    #        Use when source has h_wave_response populated (conditioning fv >= 2).
    #   7  → writes all base trial fields; read back as MH_TRIAL (block 3).
    #       Fallback when h_wave_response is NaN (conditioning fv == 1).
    _first_h = _src_trials[0].h_wave_response if _src_trials else float('nan')
    try:
        _has_h = not _math.isnan(float(_first_h))
    except (TypeError, ValueError):
        _has_h = False
    _TARGET_FV = 14 if _has_h else 7

    # ── Write ─────────────────────────────────────────────────────────────────
    write_hrs2(OUTPUT_PATH, _out_hdr, _src_trials, _src_emg, file_version=_TARGET_FV)

    print(f"\nWrote {len(_src_trials)} trials  →  {OUTPUT_PATH}")
    print(f"  Stage name    : {_out_hdr.stage_name!r}")
    print(f"  File version  : {_TARGET_FV}"
          + ("  (Control Mode — M-wave fields included)" if _TARGET_FV == 14
             else "  (MH-Recruitment compatible — no M-wave response fields)"))
    print(f"\nTo analyse in Read_H-Reflex_App.ipynb, add to RECORDING_DIRS:")
    _out_dir = _os.path.dirname(OUTPUT_PATH) or "."
    print(f'  ("<label>", "{_out_dir}", {_src_hdr.sample_rate}),')


Reading '.hrs4': HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26_20260828T102534.hrs4
  826 trials  |  source file_version=4  |  stage: 'S4'
  sample_rate=10000.0 Hz  |  subject: 'UPCP1_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26'

Wrote 826 trials  →  HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26_20260828T102534.hrs2
  Stage name    : 'Control Mode'
  File version  : 9  (Control Mode — M-wave fields included)

To analyse in Read_H-Reflex_App.ipynb, add to RECORDING_DIRS:
  ("<label>", "HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26", 10000.0),


In [ ]:
# ── Batch: Convert Conditioning Stage Files → .hrs2 ───────────────────────────
# Converts the one conditioning file (.hrs3/.hrs4/.hrs5/.hrs6) in each folder below
# with convert_conditioning_to_hrs2() (helpers.py) — same logic as the single-file
# cell above. CONVERT_DIRS uses the same (label, folder, sample_rate) tuples as
# RECORDING_DIRS in the Read notebooks; the label is only used in messages here.
#
# NOTE: with OUTPUT_ROOT = None each .hrs2 is written NEXT TO its source, so
# load_all_recordings() on that folder will then load BOTH the .hrs2 (as Control
# Mode) and the original .hrs6 stage. Set OUTPUT_ROOT to put the .hrs2 files in a
# separate folder tree (same sub-folder names) instead.
# Each file is ~3.5–5 GB and takes a few minutes; a file is written as .part first
# and renamed when complete, so an interrupted run never leaves a "finished" file.
import os as _os, glob as _glob, gc as _gc, time as _time
try:
    from helpers import convert_conditioning_to_hrs2
except ImportError:
    # Kernel imported helpers before this function existed — reload the module.
    import importlib as _importlib, helpers as _helpers
    _importlib.reload(_helpers)
    from helpers import convert_conditioning_to_hrs2

CONVERT_DIRS = [
    ("-8",  "Up-Conditioning_VNS/HRPilot-36/UPVNS1_HRPILOT-36_BOOTH2_250US_10KHZ_10-2-26",        10000.0),
    ("-7",  "Up-Conditioning_VNS/HRPilot-36/UPVNS2_HRPILOT-36_BOOTH2_250US_10KHZ_10-5-26",        10000.0),
    ("-6",  "Up-Conditioning_VNS/HRPilot-36/UPVNS3_HRPILOT-36_BOOTH2_250US_10KHZ_10-6-26",        10000.0),
    ("-5",  "Up-Conditioning_VNS/HRPilot-36/UPVNS4_HRPILOT-36_BOOTH2_250US_10KHZ_10-7-26",        10000.0),
    ("-4",  "Up-Conditioning_VNS/HRPilot-36/UPVNS5_HRPILOT-36_BOOTH2_250US_10KHZ_10-8-26",        10000.0),
    ("-3",  "Up-Conditioning_VNS/HRPilot-36/UPVNS6_HRPILOT-36_BOOTH2_250US_10KHZ_10-9-26",        10000.0),
    ("-2",  "Up-Conditioning_VNS/HRPilot-36/UPVNS7_HRPILOT-36_BOOTH2_250US_10KHZ_10-12-26",       10000.0),
    ("-1",  "Up-Conditioning_VNS/HRPilot-36/UPVNS8_HRPILOT-36_BOOTH2_250US_10KHZ_10-13-26",       10000.0),
]
OUTPUT_ROOT   = None            # None → next to each source; e.g. "Up-Conditioning_VNS_as_hrs2"
SKIP_EXISTING = True            # don't redo folders whose .hrs2 already exists
STAGE_NAME    = "Control Mode"  # None → keep the source stage name
STAGE_DESC    = None            # None → keep the source description

_conv_results = []
for _lbl, _dir, _sr in CONVERT_DIRS:
    _srcs = sorted(f for _e in ('hrs3', 'hrs4', 'hrs5', 'hrs6')
                   for f in _glob.glob(_os.path.join(_dir, f'*.{_e}')))
    if len(_srcs) != 1:
        print(f"[{_lbl}] SKIP — expected 1 conditioning file in {_dir}, found {len(_srcs)}: "
              f"{[_os.path.basename(f) for f in _srcs]}")
        continue
    _src = _srcs[0]
    _out_dir = (_dir if OUTPUT_ROOT is None
                else _os.path.join(OUTPUT_ROOT, _os.path.basename(_os.path.normpath(_dir))))
    _out = _os.path.join(_out_dir, _os.path.splitext(_os.path.basename(_src))[0] + '.hrs2')
    if SKIP_EXISTING and _os.path.isfile(_out):
        print(f"[{_lbl}] already converted — {_out}")
        _conv_results.append((_lbl, _out_dir, _sr))
        continue
    _os.makedirs(_out_dir, exist_ok=True)
    print(f"[{_lbl}] converting {_os.path.basename(_src)} …", flush=True)
    _t0 = _time.time()
    try:
        _r = convert_conditioning_to_hrs2(_src, _out + '.part',
                                          stage_name=STAGE_NAME, stage_desc=STAGE_DESC)
        _os.replace(_out + '.part', _out)
    except Exception as _err:
        print(f"[{_lbl}] FAILED: {type(_err).__name__}: {_err}")
        if _os.path.isfile(_out + '.part'):
            _os.remove(_out + '.part')
        continue
    finally:
        _gc.collect()
    print(f"[{_lbl}]   {_r['n_trials']} trials  |  source fv {_r['source_fv']} → .hrs2 fv {_r['target_fv']}"
          f"  |  {_os.path.getsize(_out) / 1e9:.2f} GB  |  {_time.time() - _t0:.0f} s  →  {_out}")
    _conv_results.append((_lbl, _out_dir, _sr))

print(f"\n{len(_conv_results)}/{len(CONVERT_DIRS)} folder(s) have a .hrs2. "
      f"To analyse them in the Read notebooks, use in RECORDING_DIRS:")
for _lbl, _d, _sr in _conv_results:
    print(f'    ("{_lbl}",  "{_d}",  {_sr}),')

# Section 9: Combine Recordings → One Recording

Joins two or more recordings of the same stage (e.g. a session split into `CM11` + `CM11_PT2`, or two VNS sessions) into one `.hrs1`–`.hrs6` file using `combine_hrs_recordings()` (helpers.py). It can also keep only some of the trials (downsampling), and works with a single recording to downsample just that one.

- **Byte-level join:** the first file's header is kept, then every file's blocks are copied in order. Trials and EMG blocks are never decoded and re-encoded, so fv≥15 Control Mode trials keep all their data (`write_hrs2` only writes up to fv14).
- Trial order = source order, so list the parts in the order they were recorded.
- The **output header is the first file's** (subject_id, session_start_time, booth snapshot, filter config, and for conditioning stages the rolling-window seed values). Later headers are dropped.
- The first file's `.settings.json` is copied to the output. Later parts' settings (e.g. a different fixed stim amplitude) are not kept, but each trial still stores its own stim amplitude.
- All sources must share extension, file_version, stage type and filtering_protocol, or the function raises an error.

**Downsampling (`COMBINE_METHOD`)** — which trials to keep:
- `None` — keep every trial.
- `"balanced"` *(recommended for pooling)* — every recording keeps the same number of trials (the smallest recording's count), spread evenly across its session. A longer session can't dominate the pooled statistics, and each session's early → late time course is kept.
- `"count"` — `COMBINE_N_TRIALS` trials in total, spread evenly over the combined trial order, so each recording contributes in proportion to its length.
- `"manual"` — `COMBINE_MANUAL` says what to keep from each recording: `"all"`, a fraction of the session like `(0.0, 0.5)` (first half), or 1-based trial numbers / inclusive ranges like `[(1, 150), 200]` (same convention as Section 7).

`COMBINE_SPACING = "random"` swaps even spacing for a random sample (fixed by `COMBINE_SEED`) in `"balanced"` / `"count"`. A dropped trial loses only its trial block: **all continuous EMG blocks are kept**, so OFFLINE reconstruction and background EMG for the kept trials are the same as in the original file. Run with `DRY_RUN = True` first to see the counts. `<output>.combine.json` records which trial numbers were kept from each source.

In [5]:
# ── Combine Recordings → One Recording ────────────────────────────────────────
# COMBINE_DIRS uses the same (label, folder, sample_rate) tuples as RECORDING_DIRS.
# Each folder must hold exactly one file with COMBINE_EXT. Order = trial order.
import os as _os, glob as _glob, time as _time
try:
    from helpers import combine_hrs_recordings
except ImportError:
    # Kernel imported helpers before this function existed — reload the module.
    import importlib as _importlib, helpers as _helpers
    _importlib.reload(_helpers)
    from helpers import combine_hrs_recordings

COMBINE_DIRS = [

    
    ("UPVNS12",  "Up-Conditioning_VNS/HRPilot-33/UPVNS12_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26",        10000.0),
    ("UPVNS13",  "Up-Conditioning_VNS/HRPilot-33/UPVNS13_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26",        10000.0),
]
COMBINED_NAME = "UPVNS12.1_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26"
COMBINE_EXT   = ".hrs6"   # ".hrs1" = MH Recruitment, ".hrs2" = Control Mode, ".hrs3"–".hrs6" = conditioning
# Output folder: next to the first source folder, named COMBINED_NAME.
COMBINED_DIR  = _os.path.join(_os.path.dirname(_os.path.normpath(COMBINE_DIRS[0][1])), COMBINED_NAME)
OVERWRITE     = False

# ── Downsampling: which trials to keep ────────────────────────────────────────
COMBINE_METHOD   = "manual"       # None | "balanced" | "count" | "manual"  (see markdown above)
COMBINE_N_TRIALS = 1123        # "count": total trials in the output
COMBINE_MANUAL   = {          # "manual": per COMBINE_DIRS label; a missing label keeps all
    "UPVNS2": "all",
    "UPVNS13": "all", 
    #"UPVNS13": (0.5, 1),          # fraction of the session → second half
    # "4": [(1, 150), 200],   # or 1-based trial numbers / inclusive ranges
}
COMBINE_SPACING  = "even"     # "even" = evenly spaced trials, "random" = random sample
COMBINE_SEED     = 0          # "random" only — same seed = same trials
DRY_RUN          = False      # True = index + show counts only, nothing is written

_srcs, _ok = [], True
for _lbl, _dir, _sr in COMBINE_DIRS:
    _found = sorted(_glob.glob(_os.path.join(_dir, '*' + COMBINE_EXT)))
    if len(_found) != 1:
        print(f"[{_lbl}] ERROR — expected 1 {COMBINE_EXT} file in {_dir}, found {len(_found)}: "
              f"{[_os.path.basename(f) for f in _found]}")
        _ok = False
    else:
        _srcs.append(_found[0])

_out = _os.path.join(COMBINED_DIR, COMBINED_NAME + COMBINE_EXT)
if _ok and _os.path.isfile(_out) and not OVERWRITE and not DRY_RUN:
    print(f"Already exists (set OVERWRITE = True to redo): {_out}")
elif _ok:
    if not DRY_RUN:
        _os.makedirs(COMBINED_DIR, exist_ok=True)
    print(f"{'Dry run' if DRY_RUN else 'Combining'} {len(_srcs)} file(s) → {_out} …", flush=True)
    _t0 = _time.time()
    _r = combine_hrs_recordings(
        _srcs, _out, method=COMBINE_METHOD, n_target=COMBINE_N_TRIALS,
        manual=[COMBINE_MANUAL.get(_lbl, "all") for _lbl, _, _ in COMBINE_DIRS],
        spacing=COMBINE_SPACING, seed=COMBINE_SEED, dry_run=DRY_RUN)
    print(f"  method={COMBINE_METHOD!r}  |  {_r['n_kept']} of {_r['n_trials']} trials kept"
          + ('' if DRY_RUN else f"  |  {_r['n_bytes'] / 1e9:.2f} GB")
          + f"  |  {_time.time() - _t0:.0f} s")
    for (_lbl, _, _), _s in zip(COMBINE_DIRS, _r['sources']):
        _k = _s['kept_trial_nums']
        _rng = f"trials {_k[0]}–{_k[-1]}" if _k else "no trials"
        print(f"  · [{_lbl}] {_os.path.basename(_s['path'])}  fv={_s['file_version']}  "
              f"{_s['n_kept']}/{_s['n_trials']} kept ({_rng})  start={_s['session_start_time']}")
        if _s['truncated_bytes']:
            print(f"      truncated final block ({_s['truncated_bytes']} bytes) not copied")
    _subj = {_s['subject_id'] for _s in _r['sources']}
    if len(_subj) > 1:
        print(f"  NOTE: sources have different subject_id {sorted(_subj)} — "
              f"output keeps the first ({_r['sources'][0]['subject_id']!r}).")
    if DRY_RUN:
        print("\nDRY_RUN = True — nothing written. Set DRY_RUN = False to write the file.")
    else:
        print(f"\nTo analyse in the Read notebooks, add to RECORDING_DIRS:")
        print(f'    ("{COMBINED_NAME.split("_")[0]}_BOTH",  "{COMBINED_DIR.replace(_os.sep, "/")}",  {COMBINE_DIRS[0][2]}),')

Combining 2 file(s) → Up-Conditioning_VNS\HRPilot-33\UPVNS12.1_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26\UPVNS12.1_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26.hrs6 …
[index_hrs_trial_blocks] UPVNS12_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26.hrs6: unknown block_id=1240 at 9466025, resynced to 9466037
[index_hrs_trial_blocks] UPVNS12_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26.hrs6: unknown block_id=1241 at 10451122, resynced to 10451134
[index_hrs_trial_blocks] UPVNS12_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26.hrs6: unknown block_id=1239 at 11432463, resynced to 11432475
[index_hrs_trial_blocks] UPVNS12_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26.hrs6: unknown block_id=1118 at 16602472, resynced to 16602484
[index_hrs_trial_blocks] UPVNS12_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26.hrs6: unknown block_id=1239 at 17718425, resynced to 17718437
[index_hrs_trial_blocks] UPVNS12_HRPILOT-33_BOOTH3_250US_10KHZ_10-30-26.hrs6: unknown block_id=1117 at 18765354, resynced to 18765366
[index_hrs_trial_blocks] UPVNS12_HRPILOT